# MCD-rPPG Dataset Preprocessing Notebook

**Complete Preprocessing Pipeline with MediaPipe Face Detection**

This notebook processes the MCD-rPPG dataset using MediaPipe for face detection and ROI extraction.
It aligns video frames with PPG/ECG signals and saves preprocessed data to the specified output paths.

## Configuration
- **VIDEOS_PATH**: `/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/`
- **FACES_PATH**: `/home/cristic/preprocessed_data/faces/`
- **LANDMARKS_PATH**: `/home/cristic/preprocessed_data/landmarks/`
- **PPG_SYNC_PATH**: `/home/cristic/preprocessed_data/ppg_sync/`
- **Output Root**: `/home/cristic/preprocessed_data/`

## ROI Definitions (MediaPipe 468 landmarks)
- `full_face`: All 468 landmarks
- `forehead`: [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10]
- `left_eye`: landmarks 22-52
- `right_eye`: landmarks 252-282
- `nose`: landmarks 1-20, 195-220
- `mouth`: landmarks 60-80, 290-320
- `left_cheek`: landmarks 0-100, 200-300
- `right_cheek`: landmarks 100-200, 300-400
- `chin`: landmarks 150-170, 370-390
- `left_iris`: landmarks 468-472
- `right_iris`: landmarks 473-477

## Workflow
1. Setup & Configuration
2. Load Database and Video List
3. Initialize MediaPipe
4. File Parsing Functions
5. ROI Extraction Functions (with Fallback)
6. PPG-Video Alignment Function
7. Video Processing with rppglib.face_utils.process_video()
8. TEST PHASE: Process Sample Videos
9. Full Dataset Processing
10. Extract and Save ROIs and Landmarks
11. Save PPG and PPG Sync Signals
12. Create Dataset Splits
13. Create Training CSV for rppglib.train.train_fold
14. Verify Output
15. Quick Test: Load and Verify One Sample

---

## 1. Setup and Configuration

In [ ]:
# Install required packages
!pip install mediapipe>=0.10.11 opencv-python-headless tqdm scikit-image numpy pandas scipy matplotlib scikit-video

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from datetime import datetime
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split
import random
import matplotlib.pyplot as plt
import skvideo.io

# Add rppglib to path
sys.path.insert(0, '/workspace/CrisChir__mcd_rppg')

import rppglib.data_utils
import rppglib.face_utils

print('All imports successful!')
print(f'MediaPipe version: {rppglib.face_utils.mp.__version__}')
print(f'rppglib imported from: {rppglib.__file__}')

In [ ]:
# Configuration
 class Config:
     # Paths
     VIDEOS_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/'
     FACES_PATH = '/home/cristic/preprocessed_data/faces/'
     LANDMARKS_PATH = '/home/cristic/preprocessed_data/landmarks/'
     PPG_SYNC_PATH = '/home/cristic/preprocessed_data/ppg_sync/'
     OUTPUT_ROOT = '/home/cristic/preprocessed_data/'
     DB_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
     
     # Processing parameters
     window_size = 256
     stride = 64
     target_size = (128, 128)
     padding = 20
     min_face_size = 64
     max_face_size = 512
     
     # Signal processing
     ppg_low_freq = 0.75
     ppg_high_freq = 4.0
     frame_rate = 30.0
     ppg_rate = 100.0
     
     # Dataset splits
     train_ratio = 0.8
     val_ratio = 0.1
     test_ratio = 0.1
     random_state = 42
     num_folds = 5
    max_ppg_plot_frames = 600  # Only plot first N frames to avoid flat line from padding
     
     # ROI definitions (MediaPipe 468 landmarks)
     rois = {
         'full_face': list(range(468)),
         'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
         'left_eye': list(range(22, 53)),
         'right_eye': list(range(252, 283)),
         'nose': list(range(1, 21)) + list(range(195, 221)),
         'mouth': list(range(60, 81)) + list(range(290, 321)),
         'left_cheek': list(range(0, 101)) + list(range(200, 301)),
         'right_cheek': list(range(100, 201)) + list(range(300, 401)),
         'chin': list(range(150, 171)) + list(range(370, 391)),
         'left_iris': list(range(468, 473)),
         'right_iris': list(range(473, 478))
     }
 
 config = Config()
 
 # Create output directories
 os.makedirs(config.FACES_PATH, exist_ok=True)
 os.makedirs(config.LANDMARKS_PATH, exist_ok=True)
 os.makedirs(config.PPG_SYNC_PATH, exist_ok=True)
 os.makedirs(config.OUTPUT_ROOT, exist_ok=True)
 
 print('Configuration initialized')
 print(f'Videos path: {config.VIDEOS_PATH}')
 print(f'Faces path: {config.FACES_PATH}')
 print(f'Landmarks path: {config.LANDMARKS_PATH}')
 print(f'PPG Sync path: {config.PPG_SYNC_PATH}')
 print(f'ROIs available: {list(config.rois.keys())}')

---

## 2. Load Database and Video List

In [ ]:
# Load database if available
db = None
if os.path.exists(config.DB_PATH):
    db = pd.read_csv(config.DB_PATH)
    print(f'Loaded database with {len(db)} rows')
    print(f'Columns: {list(db.columns)}')
    display(db.head(2))
else:
    print(f'Database not found at {config.DB_PATH}, will use file system scan')

# Get list of video files
video_files = []
if os.path.exists(config.VIDEOS_PATH):
    for file in os.listdir(config.VIDEOS_PATH):
        if file.endswith('.avi') or file.endswith('.mp4'):
            video_files.append(os.path.join(config.VIDEOS_PATH, file))
    video_files.sort()
    print(f'Found {len(video_files)} video files')
else:
    print(f'Videos path not found: {config.VIDEOS_PATH}')

# Show first few video files
for i, vf in enumerate(video_files[:5]):
    print(f'{i+1}. {os.path.basename(vf)}')

---

## 3. Initialize MediaPipe Face Landmarker

In [ ]:
# Initialize MediaPipe using rppglib
print('Initializing MediaPipe Face Landmarker...')

# We'll use the process_video function from rppglib.face_utils
# which internally handles MediaPipe initialization

# Test with a sample frame to ensure MediaPipe is working
if video_files:
    test_video = rppglib.data_utils.load_video(video_files[0])
    print(f'Loaded test video: {test_video.shape}')
    
    # Process first frame to test
    test_frame = test_video[0]
    print(f'Test frame shape: {test_frame.shape}')
    print('MediaPipe initialization test passed!')
else:
    print('No video files to test with')

---

## 4. File Parsing Functions for PPG/ECG/Meta/PPG_Sync Files

In [ ]:
def parse_pw_file(pw_path):
    '''Parse .PW file (PPG signal with timestamps)'''
    ppg_values, timestamps = [], []
    with open(pw_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    ppg_values.append(float(parts[0]))
                    timestamps.append(datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f'))
                except Exception as e:
                    print(f'Warning: Could not parse line: {line[:50]}... - {e}')
    return np.array(ppg_values), np.array(timestamps)

def parse_meta_file(meta_path):
    '''Parse meta file (frame_number timestamp)'''
    meta_data = {}
    with open(meta_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    meta_data[int(parts[0])] = datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f')
                except: pass
    return meta_data

def parse_ppg_sync_file(sync_path):
    '''Parse PPG sync file (frame_number sync_value) - Already aligned with video!'''
    sync_data = {}
    with open(sync_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    sync_data[int(parts[0])] = float(parts[1])
                except: pass
    return sync_data

def parse_ppg_sync_directory(sync_path, num_frames):
    '''
    Parse PPG sync values from directory and return array aligned with video frames.
    PPG sync files are already aligned with video frames!
    Each line: frame_number sync_value
    Returns: array of sync values for each frame
    '''
    sync_data = parse_ppg_sync_file(sync_path)
    # Create array of sync values for each frame
    ppg_sync = np.zeros(num_frames)
    for frame_num, sync_val in sync_data.items():
        if 0 <= frame_num < num_frames:
            ppg_sync[frame_num] = sync_val
    return ppg_sync

def find_matching_files(video_path):
    '''Find PPG, meta, sync, and ECG files matching a video file'''
    base_name = os.path.splitext(os.path.basename(video_path))[0]
    dataset_root = os.path.dirname(os.path.dirname(video_path))
    
    files = {
        'ppg': None,
        'meta': None,
        'ppg_sync': None,
        'ecg': None
    }
    
    # Try to find PPG file
    ppg_dir = os.path.join(dataset_root, 'ppg')
    if os.path.exists(ppg_dir):
        for f in os.listdir(ppg_dir):
            if f.startswith(base_name) and f.endswith('.PW'):
                files['ppg'] = os.path.join(ppg_dir, f)
                break
    
    # Try to find meta file
    meta_dir = os.path.join(dataset_root, 'meta')
    if os.path.exists(meta_dir):
        for f in os.listdir(meta_dir):
            if f.startswith(base_name) and f.endswith('.txt'):
                files['meta'] = os.path.join(meta_dir, f)
                break
    
    # Try to find ppg_sync file
    sync_dir = os.path.join(dataset_root, 'ppg_sync')
    if os.path.exists(sync_dir):
        for f in os.listdir(sync_dir):
            if f.startswith(base_name) and f.endswith('.txt'):
                files['ppg_sync'] = os.path.join(sync_dir, f)
                break
    
    # Try to find ECG file
    ecg_dir = os.path.join(dataset_root, 'ecg')
    if os.path.exists(ecg_dir):
        for f in os.listdir(ecg_dir):
            if f.startswith(base_name):
                files['ecg'] = os.path.join(ecg_dir, f)
                break
    
    return files

---

## 5. ROI Extraction Functions (with Fallback) - MUST RUN BEFORE VIDEO PROCESSING

In [ ]:
# Fallback ROI extraction function in case rppglib doesn't have extract_roi
def fallback_extract_roi(video, landmarks, roi_name):
    '''
    Fallback implementation for ROI extraction.
    Used when rppglib.face_utils.extract_roi is not available.
    
    ROI definitions for MediaPipe 468-point model:
    - 'full_face': All 468 landmarks
    - 'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10]
    - 'left_eye': landmarks 22-41 + 220-239
    - 'right_eye': landmarks 42-61 + 242-261
    - 'nose': landmarks 1-19 + 195-219
    - 'mouth': landmarks 60-79 + 290-309
    - 'left_cheek': landmarks 0-99 + 200-299
    - 'right_cheek': landmarks 100-199 + 300-399
    - 'chin': landmarks 150-170 + 370-390
    - 'left_iris': landmarks 468-472
    - 'right_iris': landmarks 473-477
    '''
    # Define ROI landmark indices for MediaPipe 468-point model
    roi_indices = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 42)) + list(range(220, 240)),
        'right_eye': list(range(42, 62)) + list(range(242, 262)),
        'nose': list(range(1, 20)) + list(range(195, 220)),
        'mouth': list(range(60, 80)) + list(range(290, 310)),
        'left_cheek': list(range(0, 100)) + list(range(200, 300)),
        'right_cheek': list(range(100, 200)) + list(range(300, 400)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }
    
    if roi_name not in roi_indices:
        raise ValueError(f'Unknown ROI: {roi_name}. Available: {list(roi_indices.keys())}')
    
    indices = roi_indices[roi_name]
    
    # Extract ROI from each frame
    roi_frames = []
    for frame_idx in range(video.shape[0]):
        frame = video[frame_idx]
        frame_landmarks = landmarks[frame_idx]
        
        # Get ROI landmarks
        roi_landmarks = frame_landmarks[indices]
        
        # Find bounding box of ROI
        x_min = roi_landmarks[:, 0].min()
        x_max = roi_landmarks[:, 0].max()
        y_min = roi_landmarks[:, 1].min()
        y_max = roi_landmarks[:, 1].max()
        
        # Add padding
        pad = 10
        x_min = max(0, int(x_min) - pad)
        x_max = min(frame.shape[1], int(x_max) + pad)
        y_min = max(0, int(y_min) - pad)
        y_max = min(frame.shape[0], int(y_max) + pad)
        
        # Extract ROI
        roi_frame = frame[y_min:y_max, x_min:x_max]
        roi_frames.append(roi_frame)
    
    return np.array(roi_frames)

# Safe wrapper for extract_roi
def safe_extract_roi(video, landmarks, roi_name):
    '''
    Safe wrapper for ROI extraction that handles different API versions.
    '''
    try:
        # Try using rppglib's extract_roi first
        return rppglib.face_utils.extract_roi(video, landmarks, roi_name)
    except AttributeError:
        # Fallback to our implementation
        print(f'  Note: rppglib.face_utils.extract_roi not available, using fallback implementation')
        return fallback_extract_roi(video, landmarks, roi_name)
    except Exception as e:
        print(f'  Error in safe_extract_roi: {e}')
        raise

---

## 6. PPG-Video Alignment Function

In [ ]:
def align_ppg_with_video(ppg_values, ppg_timestamps, meta_data, frame_rate=30.0):
    '''
    Align PPG signal with video frames using timestamp interpolation.
    
    Args:
        ppg_values: Array of PPG values
        ppg_timestamps: Array of datetime objects for PPG
        meta_data: Dict mapping frame_number to timestamp
        frame_rate: Video frame rate
    
    Returns:
        aligned_ppg: Array of PPG values aligned with video frames
    '''
    if len(meta_data) == 0 or len(ppg_timestamps) == 0:
        return np.zeros(len(meta_data))
    
    # Convert timestamps to seconds
    first_ppg = ppg_timestamps[0]
    first_meta = list(meta_data.values())[0]
    
    ppg_seconds = [(ts - first_ppg).total_seconds() for ts in ppg_timestamps]
    sorted_frames = sorted(meta_data.keys())
    meta_seconds = [(meta_data[f] - first_meta).total_seconds() for f in sorted_frames]
    
    # Interpolate PPG values at video frame timestamps
    interp_func = interp1d(ppg_seconds, ppg_values, kind='linear', fill_value='extrapolate')
    aligned_ppg = interp_func(meta_seconds)
    
    return aligned_ppg

def preprocess_ppg(ppg, low_freq=0.75, high_freq=4.0, fs=100.0):
    '''Apply bandpass filter and normalize PPG signal'''
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, ppg)
    return (filtered - filtered.mean()) / filtered.std()

def simple_ratio_align(ppg, video_length, ppg_rate=100.0, video_rate=30.0):
    '''
    Simple ratio-based alignment (fallback when timestamps are unavailable).
    Uses the ratio: ppg_samples_per_video_frame = ppg_rate / video_rate
    '''
    ratio = ppg_rate / video_rate
    aligned_ppg = []
    for frame_idx in range(video_length):
        ppg_start = int(frame_idx * ratio)
        ppg_end = int((frame_idx + 1) * ratio)
        if ppg_end <= len(ppg):
            # Average PPG values for this frame
            frame_ppg = ppg[ppg_start:ppg_end]
            aligned_ppg.append(np.mean(frame_ppg))
        else:
            aligned_ppg.append(0.0)
    return np.array(aligned_ppg)

def preprocess_ppg_sync(ppg_sync, low_freq=0.75, high_freq=4.0, fs=30.0):
    '''
    Preprocess PPG sync signal (already aligned with video at 30 FPS).
    PPG sync is at video frame rate, not PPG sensor rate.
    '''
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, ppg_sync)
    return (filtered - filtered.mean()) / filtered.std()

---

## 7. Video Processing Functions (with API Compatibility)

In [ ]:
# Wrapper function for process_video to handle different API versions
def safe_process_video(video, min_face_size=64, max_face_size=512, running_mode=None):
    '''
    Wrapper for rppglib.face_utils.process_video that handles different API versions.
    Some versions may not accept min_face_size/max_face_size parameters.
    '''
    try:
        # Try with all parameters (new API)
        if running_mode is not None:
            return rppglib.face_utils.process_video(
                video, min_face_size=min_face_size, max_face_size=max_face_size, running_mode=running_mode
            )
        else:
            return rppglib.face_utils.process_video(
                video, min_face_size=min_face_size, max_face_size=max_face_size
            )
    except TypeError as e:
        # Fallback: try without min_face_size/max_face_size (old API)
        if 'min_face_size' in str(e):
            print(f'  Note: process_video does not support min_face_size/max_face_size, using defaults')
            if running_mode is not None:
                return rppglib.face_utils.process_video(video, running_mode=running_mode)
            else:
                return rppglib.face_utils.process_video(video)
        else:
            raise
    except Exception as e:
        raise RuntimeError(f'Failed to process video: {e}')

def process_single_video(video_path, config, use_roi=None):
    '''
    Process a single video file using rppglib.face_utils.process_video()
    
    Args:
        video_path: Path to video file
        config: Configuration object
        use_roi: ROI name to extract (None for full face)
    
    Returns:
        dict with processed_video, landmarks, ppg_sync, and metadata
    '''
    result = {
        'video_path': video_path,
        'success': False,
        'error': None,
        'processed_video': None,
        'landmarks': None,
        'roi_video': None,
        'aligned_ppg': None,
        'ppg_sync': None,
        'metadata': {}
    }
    
    try:
        # Load video using rppglib
        video = rppglib.data_utils.load_video(video_path)
        result['metadata']['original_shape'] = video.shape
        result['metadata']['num_frames'] = len(video)
        
        # Process video with MediaPipe using rppglib (with safe wrapper)
        processed_video, landmarks = safe_process_video(
            video,
            min_face_size=config.min_face_size,
            max_face_size=config.max_face_size
        )
        
        result['processed_video'] = processed_video
        result['landmarks'] = landmarks
        result['metadata']['processed_shape'] = processed_video.shape
        
        # Extract ROI if specified
        if use_roi and use_roi in config.rois:
            roi_video = safe_extract_roi(processed_video, landmarks, use_roi)
            result['roi_video'] = roi_video
            result['metadata']['roi_shape'] = roi_video.shape
        
        # Find matching PPG/meta/sync files
        matching_files = find_matching_files(video_path)
        result['metadata']['matching_files'] = matching_files
        
        # Load PPG sync (ALREADY ALIGNED with video frames!)
        if matching_files['ppg_sync'] and os.path.exists(matching_files['ppg_sync']):
            ppg_sync_raw = parse_ppg_sync_directory(
                matching_files['ppg_sync'], len(processed_video)
            )
            # Preprocess PPG sync signal
            result['ppg_sync'] = preprocess_ppg_sync(
                ppg_sync_raw, config.ppg_low_freq, config.ppg_high_freq, config.frame_rate
            )
            result['metadata']['ppg_sync_length'] = len(result['ppg_sync'])
            result['metadata']['ppg_sync_available'] = True
            print(f'  Loaded PPG sync: {len(ppg_sync_raw)} frames')
        else:
            result['metadata']['ppg_sync_available'] = False
            print(f'  Warning: No PPG sync file found for {video_path}')
        
        # Align PPG with video if available (for reference)
        if matching_files['ppg'] and matching_files['meta']:
            ppg_values, ppg_timestamps = parse_pw_file(matching_files['ppg'])
            meta_data = parse_meta_file(matching_files['meta'])
            
            if len(ppg_values) > 0 and len(meta_data) > 0:
                aligned_ppg = align_ppg_with_video(
                    ppg_values, ppg_timestamps, meta_data, config.frame_rate
                )
                result['aligned_ppg'] = preprocess_ppg(aligned_ppg, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
                result['metadata']['ppg_length'] = len(aligned_ppg)
            else:
                # Fallback to simple ratio-based alignment
                ppg_values_simple = parse_pw_file(matching_files['ppg'])[0]
                if len(ppg_values_simple) > 0:
                    aligned_ppg_simple = simple_ratio_align(
                        ppg_values_simple, len(processed_video), config.ppg_rate, config.frame_rate
                    )
                    result['aligned_ppg'] = preprocess_ppg(aligned_ppg_simple, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
                    result['metadata']['ppg_length'] = len(aligned_ppg_simple)
                    result['metadata']['alignment_method'] = 'ratio'
        
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)
        result['success'] = False
        import traceback
        print(f'Error processing {video_path}:')
        traceback.print_exc()
    
    return result

---

## 8. TEST PHASE: Process Sample Videos

In [ ]:
# Test with a few videos first
 print('=' * 80)
 print('TEST PHASE: Processing sample videos')
 print('=' * 80)
 
 test_limit = min(3, len(video_files))
 test_results = []
 
 for i in range(test_limit):
     video_path = video_files[i]
     print(f'\nProcessing test video {i+1}/{test_limit}: {os.path.basename(video_path)}')
     
     result = process_single_video(video_path, config, use_roi=None)
     test_results.append(result)
     
     if result['success']:
         print(f'  Success!')
         print(f'    Original: {result["metadata"].get("original_shape")}')
         print(f'    Processed: {result["metadata"].get("processed_shape")}')
         if result['roi_video'] is not None:
             print(f'    ROI (forehead): {result["metadata"].get("roi_shape")}')
         if result['ppg_sync'] is not None:
             print(f'    PPG Sync: {result["metadata"].get("ppg_sync_length")} frames (ALREADY ALIGNED)')
         if result['aligned_ppg'] is not None:
             print(f'    Aligned PPG: {result["metadata"].get("ppg_length")} samples')
         print(f'    Landmarks: {result["landmarks"].shape if result["landmarks"] is not None else None}')
     else:
         print(f'  Failed: {result["error"]}')
 
 print(f'\nTest phase complete: {sum(1 for r in test_results if r["success"])}/{test_limit} successful')

In [ ]:
# Enhanced visualization with all ROIs and all PPG types
if test_results and test_results[0]['success']:
    result = test_results[0]
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    processed_video = result['processed_video']
    landmarks = result['landmarks']
    ppg_sync = result['ppg_sync']
    aligned_ppg = result['aligned_ppg']
    
    print(f'\n{"="*80}')
    print(f'ENHANCED VISUALIZATION for: {video_id}')
    print(f'{"="*80}')
    
    # ========================================================================
    # SECTION 1: Video Frames
    # ========================================================================
    print('\n1. Video Frames:')
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # First, middle, last frames
    frame_indices = [0, len(processed_video) // 2, len(processed_video) - 1]
    for idx, ax in zip(frame_indices, axes):
        ax.imshow(processed_video[idx])
        ax.set_title(f'Frame {idx}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    # ========================================================================
    # SECTION 2: All ROIs (First Frame)
    # ========================================================================
    print('\n2. All ROIs - First Frame:')
    # Extract all ROIs for first frame only for visualization
    first_frame = processed_video[0]
    first_landmarks = landmarks[0]
    
    # Calculate grid size for ROI display
    num_rois = len(config.rois)
    cols = 4
    rows = (num_rois + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes_flat = axes.flatten() if isinstance(axes, np.ndarray) else [axes]
    
    for roi_idx, (roi_name, roi_indices) in enumerate(config.rois.items()):
        ax = axes_flat[roi_idx]
        try:
            # Extract ROI for first frame
            roi_landmarks = first_landmarks[roi_indices]
            x_min = max(0, int(roi_landmarks[:, 0].min()) - 10)
            x_max = min(first_frame.shape[1], int(roi_landmarks[:, 0].max()) + 10)
            y_min = max(0, int(roi_landmarks[:, 1].min()) - 10)
            y_max = min(first_frame.shape[0], int(roi_landmarks[:, 1].max()) + 10)
            roi_img = first_frame[y_min:y_max, x_min:x_max]
            ax.imshow(roi_img)
            ax.set_title(f'{roi_name}: {roi_img.shape[1]}x{roi_img.shape[0]}')
            ax.axis('off')
        except Exception as e:
            ax.set_title(f'{roi_name}: Error')
            ax.axis('off')
    
    # Hide unused subplots
    for idx in range(num_rois, len(axes_flat)):
        axes_flat[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # ========================================================================
    # SECTION 3: PPG Signals Comparison
    # ========================================================================
    print('\n3. PPG Signals Comparison:')
    
    # Determine valid range (avoid flat line from padding)
    max_plot = 600  # Only plot first 600 frames
    
    if ppg_sync is not None:
        ppg_sync_plot = ppg_sync[:max_plot]
        print(f'   PPG Sync: Plotting {len(ppg_sync_plot)} frames')
    else:
        ppg_sync_plot = None
    
    if aligned_ppg is not None:
        aligned_ppg_plot = aligned_ppg[:max_plot]
        print(f'   Aligned PPG: Plotting {len(aligned_ppg_plot)} samples')
    else:
        aligned_ppg_plot = None
    
    # Create comparison plot
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    # PPG Sync (from ppg_sync file - already aligned with video)
    if ppg_sync_plot is not None:
        axes[0].plot(ppg_sync_plot, label='PPG Sync (from ppg_sync)', color='blue', alpha=0.8)
        axes[0].set_title('PPG Sync Signal (from ppg_sync file) - ALIGNED with video frames', fontsize=12, fontweight='bold')
        axes[0].set_xlabel('Frame Index')
        axes[0].set_ylabel('PPG Value (normalized)')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()
    else:
        axes[0].set_title('PPG Sync: Not available')
        axes[0].axis('off')
    
    # Aligned PPG (from .PW file - interpolated to video frames)
    if aligned_ppg_plot is not None:
        axes[1].plot(aligned_ppg_plot, label='Aligned PPG (from .PW)', color='red', alpha=0.8)
        axes[1].set_title('Aligned PPG Signal (from .PW file) - Interpolated to video frames', fontsize=12, fontweight='bold')
        axes[1].set_xlabel('Sample Index')
        axes[1].set_ylabel('PPG Value (normalized)')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()
    else:
        axes[1].set_title('Aligned PPG: Not available')
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # ========================================================================
    # SECTION 4: Overlay Comparison (if both PPG types available)
    # ========================================================================
    if ppg_sync_plot is not None and aligned_ppg_plot is not None:
        print('\n4. PPG Signals Overlay (First 600 samples):')
        # Resample aligned PPG to match video frame rate for comparison
        min_len = min(len(ppg_sync_plot), len(aligned_ppg_plot))
        if min_len > 0:
            # Normalize both for comparison
            ppg_sync_norm = (ppg_sync_plot[:min_len] - ppg_sync_plot[:min_len].mean()) / (ppg_sync_plot[:min_len].std() + 1e-9)
            aligned_ppg_norm = (aligned_ppg_plot[:min_len] - aligned_ppg_plot[:min_len].mean()) / (aligned_ppg_plot[:min_len].std() + 1e-9)
            
            fig, ax = plt.subplots(1, 1, figsize=(14, 6))
            ax.plot(ppg_sync_norm, label='PPG Sync (video-aligned)', color='blue', linewidth=1.5)
            ax.plot(aligned_ppg_norm, label='PPG from .PW (interpolated)', color='red', linewidth=1.5, alpha=0.7)
            ax.set_title(f'PPG Signals Overlay - First {min_len} samples', fontsize=14, fontweight='bold')
            ax.set_xlabel('Frame/Sample Index')
            ax.set_ylabel('PPG Value (normalized)')
            ax.grid(True, alpha=0.3)
            ax.legend()
            plt.tight_layout()
            plt.show()
            
            # Calculate correlation
            correlation = np.corrcoef(ppg_sync_norm, aligned_ppg_norm)[0, 1]
            print(f'   Correlation between PPG Sync and Aligned PPG: {correlation:.4f}')
    
    # ========================================================================
    # SECTION 5: Statistics Summary
    # ========================================================================
    print('\n5. Signal Statistics:')
    if ppg_sync is not None:
        print(f'   PPG Sync: mean={ppg_sync.mean():.4f}, std={ppg_sync.std():.4f}, min={ppg_sync.min():.4f}, max={ppg_sync.max():.4f}')
    if aligned_ppg is not None:
        print(f'   Aligned PPG: mean={aligned_ppg.mean():.4f}, std={aligned_ppg.std():.4f}, min={aligned_ppg.min():.4f}, max={aligned_ppg.max():.4f}')
    print(f'   Video: {len(processed_video)} frames, shape={processed_video.shape[1:3]}')
    print(f'   Landmarks: {landmarks.shape}')
    
    print(f'\n{"="*80}')
    print('Visualization complete!')
    print(f'{"="*80}\n')

else:
    print('No successful test results to visualize')

---

## 9. Full Dataset Processing

In [ ]:
# Process all videos
print('=' * 80)
print('FULL PROCESSING: Processing all videos')
print('=' * 80)

# Create output directories for each ROI
for roi_name in config.rois:
    roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
    os.makedirs(roi_dir, exist_ok=True)

all_results = []
success_count = 0
failure_count = 0
ppg_sync_count = 0

# Process videos with progress bar
for i, video_path in enumerate(tqdm(video_files, desc='Processing videos')):
    result = process_single_video(video_path, config, use_roi=None)
    all_results.append(result)
    
    if result['success']:
        success_count += 1
        if result['metadata'].get('ppg_sync_available', False):
            ppg_sync_count += 1
    else:
        failure_count += 1
        print(f'Failed: {os.path.basename(video_path)} - {result["error"]}')

print(f'\nProcessing complete: {success_count} successful, {failure_count} failed out of {len(video_files)} total')
print(f'PPG Sync available for: {ppg_sync_count}/{success_count} videos')

---

## 10. Extract and Save ROIs and Landmarks

In [ ]:
# Extract and save ROIs for all successful results
print('\nExtracting and saving ROIs...')

roi_data = {roi: [] for roi in config.rois}
landmarks_data = []
metadata_list = []

for i, result in enumerate(all_results):
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    processed_video = result['processed_video']
    landmarks = result['landmarks']
    
    # Save landmarks
    if landmarks is not None:
        landmarks_file = os.path.join(config.LANDMARKS_PATH, f'{video_id}_landmarks.npy')
        np.save(landmarks_file, landmarks)
        landmarks_data.append({
            'video_id': video_id,
            'landmarks_file': landmarks_file,
            'num_frames': len(landmarks),
            'shape': landmarks.shape
        })
    
    # Extract and save all ROIs
    for roi_name in config.rois:
        try:
            roi_video = safe_extract_roi(processed_video, landmarks, roi_name)
            roi_data[roi_name].append({
                'video_id': video_id,
                'roi_video': roi_video,
                'landmarks': landmarks
            })
            
            # Save individual ROI video
            roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
            roi_file = os.path.join(roi_dir, f'{video_id}.npy')
            np.save(roi_file, roi_video)
            
        except Exception as e:
            print(f'Warning: Could not extract ROI {roi_name} for {video_id}: {e}')
    
    # Save metadata
    metadata_list.append({
        'video_id': video_id,
        'video_path': result['video_path'],
        'num_frames': result['metadata'].get('num_frames'),
        'processed_shape': result['metadata'].get('processed_shape'),
        'has_ppg': result['aligned_ppg'] is not None,
        'ppg_length': result['metadata'].get('ppg_length'),
        'has_ppg_sync': result['metadata'].get('ppg_sync_available', False),
        'ppg_sync_length': result['metadata'].get('ppg_sync_length'),
        'alignment_method': result['metadata'].get('alignment_method', 'timestamp')
    })

print(f'Saved ROIs for {len(roi_data["full_face"])} videos')
print(f'Saved landmarks for {len(landmarks_data)} videos')

In [ ]:
# Save combined ROI arrays
print('\nSaving combined ROI arrays...')

for roi_name, roi_list in roi_data.items():
    if roi_list:
        all_roi_videos = [item['roi_video'] for item in roi_list]
        # Pad videos to same length if needed
        max_frames = max(v.shape[0] for v in all_roi_videos)
        padded_videos = []
        for v in all_roi_videos:
            if v.shape[0] < max_frames:
                # Pad with zeros
                pad_shape = (max_frames - v.shape[0], *v.shape[1:])
                padding = np.zeros(pad_shape, dtype=v.dtype)
                v_padded = np.concatenate([v, padding], axis=0)
                padded_videos.append(v_padded)
            else:
                padded_videos.append(v)
        
        combined_roi = np.array(padded_videos)
        combined_file = os.path.join(config.OUTPUT_ROOT, f'{roi_name}_all.npy')
        np.save(combined_file, combined_roi)
        print(f'  {roi_name}: {combined_roi.shape} -> {combined_file}')

# Save metadata
metadata_df = pd.DataFrame(metadata_list)
metadata_file = os.path.join(config.OUTPUT_ROOT, 'metadata.csv')
metadata_df.to_csv(metadata_file, index=False)
print(f'\nSaved metadata: {metadata_file}')
print(metadata_df.head())

---

## 11. Save PPG and PPG Sync Signals

In [ ]:
# Save PPG and PPG Sync signals
print('\nSaving PPG and PPG Sync signals...')

ppg_data = []
ppg_sync_data = []

for i, result in enumerate(all_results):
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    
    # Save PPG sync (ALREADY ALIGNED with video frames)
    if result['ppg_sync'] is not None:
        ppg_sync_file = os.path.join(config.PPG_SYNC_PATH, f'{video_id}.npy')
        os.makedirs(os.path.dirname(ppg_sync_file), exist_ok=True)
        np.save(ppg_sync_file, result['ppg_sync'])
        ppg_sync_data.append({
            'video_id': video_id,
            'ppg_sync_file': ppg_sync_file,
            'ppg_sync_length': len(result['ppg_sync']),
            'frame_rate': config.frame_rate
        })
        print(f'  Saved PPG Sync: {video_id} -> {ppg_sync_file} ({len(result["ppg_sync"])} frames)')
    
    # Save aligned PPG (from .PW files) for reference
    if result['aligned_ppg'] is not None:
        ppg_file = os.path.join(config.OUTPUT_ROOT, 'ppg', f'{video_id}.npy')
        os.makedirs(os.path.dirname(ppg_file), exist_ok=True)
        np.save(ppg_file, result['aligned_ppg'])
        ppg_data.append({
            'video_id': video_id,
            'ppg_file': ppg_file,
            'ppg_length': len(result['aligned_ppg']),
            'alignment_method': result['metadata'].get('alignment_method', 'timestamp')
        })

ppg_df = pd.DataFrame(ppg_data)
if len(ppg_df) > 0:
    ppg_metadata_file = os.path.join(config.OUTPUT_ROOT, 'ppg_metadata.csv')
    ppg_df.to_csv(ppg_metadata_file, index=False)
    print(f'\nSaved PPG metadata: {ppg_metadata_file}')
    print(f'Total PPG signals saved: {len(ppg_df)}')
else:
    print('\nNo PPG signals to save')

ppg_sync_df = pd.DataFrame(ppg_sync_data)
if len(ppg_sync_df) > 0:
    ppg_sync_metadata_file = os.path.join(config.OUTPUT_ROOT, 'ppg_sync_metadata.csv')
    ppg_sync_df.to_csv(ppg_sync_metadata_file, index=False)
    print(f'Saved PPG Sync metadata: {ppg_sync_metadata_file}')
    print(f'Total PPG Sync signals saved: {len(ppg_sync_df)}')
else:
    print('No PPG Sync signals to save')

---

## 12. Create Dataset Splits

In [ ]:
# Create train/val/test splits by video ID
print('\nCreating dataset splits...')

video_ids = [os.path.splitext(os.path.basename(r['video_path']))[0] for r in all_results if r['success']]

# Split by video ID (stratified by subject if available)
train_ids, test_ids = train_test_split(
    video_ids,
    test_size=config.test_ratio + config.val_ratio,
    random_state=config.random_state
)
val_ids, test_ids = train_test_split(
    test_ids,
    test_size=config.test_ratio / (config.test_ratio + config.val_ratio),
    random_state=config.random_state
)

# Save splits
splits = {
    'train': train_ids,
    'val': val_ids,
    'test': test_ids
}

for split_name, split_ids in splits.items():
    split_file = os.path.join(config.OUTPUT_ROOT, f'{split_name}_videos.txt')
    with open(split_file, 'w') as f:
        for vid in split_ids:
            f.write(f'{vid}\n')
    print(f'  {split_name}: {len(split_ids)} videos -> {split_file}')
    
    # Also save as numpy arrays
    np.save(os.path.join(config.OUTPUT_ROOT, f'{split_name}_videos.npy'), np.array(split_ids))

# Create fold assignments for cross-validation
print('\nCreating fold assignments for cross-validation...')
from sklearn.model_selection import KFold
kf = KFold(n_splits=config.num_folds, shuffle=True, random_state=config.random_state)
fold_assignments = {}
for fold_idx, (train_fold, val_fold) in enumerate(kf.split(video_ids)):
    for vid in train_fold:
        fold_assignments[video_ids[vid]] = fold_idx
print(f'Assigned {len(fold_assignments)} videos to {config.num_folds} folds')

---

## 13. Create Training CSV for rppglib.train.train_fold

In [ ]:
# Create training CSV compatible with rppglib.train.train_fold
# Format: file,video_dtype,video_shape,video_min,video_max,video_mean,video_std,
#         ppg_dtype,ppg_shape,ppg_min,ppg_max,ppg_mean,ppg_std,
#         landmarks_dtype,landmarks_shape,landmarks_min,landmarks_max,landmarks_mean,landmarks_std,
#         face_square,video_duration,patient_id,fold

print('\nCreating training CSV for rppglib.train.train_fold...')
print('Using PPG Sync signals (already aligned with video)')

# Collect data for training CSV
training_data = []

for result in all_results:
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    processed_video = result['processed_video']
    landmarks = result['landmarks']
    ppg_sync = result['ppg_sync']
    
    # Skip if no PPG sync available
    if ppg_sync is None:
        print(f'  Skipping {video_id}: No PPG sync available')
        continue
    
    # Create .npz file for this sample (required by rppglib.train.rPPG_Dataset)
    npz_file = os.path.join(config.OUTPUT_ROOT, 'train_data', f'{video_id}.npz')
    os.makedirs(os.path.dirname(npz_file), exist_ok=True)
    
    # Save as .npz with video, ppg, landmarks
    np.savez(npz_file, 
             video=processed_video, 
             ppg=ppg_sync, 
             landmarks=landmarks)
    
    # Calculate statistics
    video_stats = {
        'dtype': str(processed_video.dtype),
        'shape': processed_video.shape,
        'min': float(processed_video.min()),
        'max': float(processed_video.max()),
        'mean': float(processed_video.mean()),
        'std': float(processed_video.std())
    }
    
    ppg_stats = {
        'dtype': str(ppg_sync.dtype),
        'shape': ppg_sync.shape,
        'min': float(ppg_sync.min()),
        'max': float(ppg_sync.max()),
        'mean': float(ppg_sync.mean()),
        'std': float(ppg_sync.std())
    }
    
    landmarks_stats = {
        'dtype': str(landmarks.dtype),
        'shape': landmarks.shape,
        'min': float(landmarks.min()),
        'max': float(landmarks.max()),
        'mean': float(landmarks.mean()),
        'std': float(landmarks.std())
    }
    
    # Calculate face square (approximate face area)
    if len(landmarks) > 0:
        x_min, x_max = landmarks[:, :, 0].min(), landmarks[:, :, 0].max()
        y_min, y_max = landmarks[:, :, 1].min(), landmarks[:, :, 1].max()
        face_square = int((x_max - x_min) * (y_max - y_min))
    else:
        face_square = 0
    
    # Video duration in seconds
    video_duration = len(processed_video) / config.frame_rate
    
    # Extract patient_id from video_id if possible
    patient_id = video_id.split('_')[0] if '_' in video_id else video_id
    
    # Get fold assignment
    fold = fold_assignments.get(video_id, 0)
    
    training_data.append({
        'file': npz_file,
        'video_dtype': video_stats['dtype'],
        'video_shape': str(video_stats['shape']),
        'video_min': video_stats['min'],
        'video_max': video_stats['max'],
        'video_mean': video_stats['mean'],
        'video_std': video_stats['std'],
        'ppg_dtype': ppg_stats['dtype'],
        'ppg_shape': str(ppg_stats['shape']),
        'ppg_min': ppg_stats['min'],
        'ppg_max': ppg_stats['max'],
        'ppg_mean': ppg_stats['mean'],
        'ppg_std': ppg_stats['std'],
        'landmarks_dtype': landmarks_stats['dtype'],
        'landmarks_shape': str(landmarks_stats['shape']),
        'landmarks_min': landmarks_stats['min'],
        'landmarks_max': landmarks_stats['max'],
        'landmarks_mean': landmarks_stats['mean'],
        'landmarks_std': landmarks_stats['std'],
        'face_square': face_square,
        'video_duration': video_duration,
        'patient_id': patient_id,
        'fold': fold
    })
    
    print(f'  Created: {video_id} -> {npz_file} (fold {fold})')

# Create DataFrame and save CSV
train_df = pd.DataFrame(training_data)

# Save as mcd_rppg.csv (standard name for rppglib)
csv_file = os.path.join(config.OUTPUT_ROOT, 'mcd_rppg.csv')
train_df.to_csv(csv_file, index=False)

print(f'\nSaved training CSV: {csv_file}')
print(f'Total samples: {len(train_df)}')
print(f'Columns: {list(train_df.columns)}')
display(train_df.head())

In [ ]:
# Also create a simplified CSV for direct use with train_fold
# This version just has file and fold columns
print('\nCreating simplified training CSV...')

simple_training_data = []
for result in all_results:
    if not result['success']:
        continue
    
    video_id = os.path.splitext(os.path.basename(result['video_path']))[0]
    ppg_sync = result['ppg_sync']
    
    if ppg_sync is None:
        continue
    
    npz_file = os.path.join(config.OUTPUT_ROOT, 'train_data', f'{video_id}.npz')
    fold = fold_assignments.get(video_id, 0)
    
    simple_training_data.append({
        'file': npz_file,
        'fold': fold
    })

simple_train_df = pd.DataFrame(simple_training_data)
simple_csv_file = os.path.join(config.OUTPUT_ROOT, 'train_folds.csv')
simple_train_df.to_csv(simple_csv_file, index=False)

print(f'Saved simplified training CSV: {simple_csv_file}')
print(f'Total samples: {len(simple_train_df)}')
display(simple_train_df.head())

---

## 14. Verify Output

In [ ]:
# Verify all output files
print('\n' + '=' * 80)
print('VERIFICATION: Checking output files')
print('=' * 80)

# Check output directories
output_dirs = [
    config.OUTPUT_ROOT,
    config.FACES_PATH,
    config.LANDMARKS_PATH,
    config.PPG_SYNC_PATH,
    os.path.join(config.OUTPUT_ROOT, 'ppg'),
    os.path.join(config.OUTPUT_ROOT, 'train_data')
]

for d in output_dirs:
    if os.path.exists(d):
        files = os.listdir(d)
        print(f'✓ {d}: {len(files)} files')
    else:
        print(f'✗ {d}: NOT FOUND')

# Check ROI directories
print(f'\nROI directories:')
for roi_name in config.rois:
    roi_dir = os.path.join(config.OUTPUT_ROOT, roi_name)
    if os.path.exists(roi_dir):
        files = os.listdir(roi_dir)
        print(f'  {roi_name}: {len(files)} files')
    else:
        print(f'  {roi_name}: NOT FOUND')

# Check key files
key_files = [
    os.path.join(config.OUTPUT_ROOT, 'metadata.csv'),
    os.path.join(config.OUTPUT_ROOT, 'ppg_metadata.csv'),
    os.path.join(config.OUTPUT_ROOT, 'ppg_sync_metadata.csv'),
    os.path.join(config.OUTPUT_ROOT, 'train_videos.txt'),
    os.path.join(config.OUTPUT_ROOT, 'val_videos.txt'),
    os.path.join(config.OUTPUT_ROOT, 'test_videos.txt'),
    os.path.join(config.OUTPUT_ROOT, 'mcd_rppg.csv'),
    os.path.join(config.OUTPUT_ROOT, 'train_folds.csv')
]

print(f'\nKey files:')
for f in key_files:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024 * 1024)
        print(f'  ✓ {os.path.basename(f)}: {size:.2f} MB')
    else:
        print(f'  ✗ {os.path.basename(f)}: NOT FOUND')

# Count .npz files
train_data_dir = os.path.join(config.OUTPUT_ROOT, 'train_data')
if os.path.exists(train_data_dir):
    npz_files = [f for f in os.listdir(train_data_dir) if f.endswith('.npz')]
    print(f'\nTrain data (.npz files): {len(npz_files)} files')
else:
    print('\nTrain data directory not found')

# Summary statistics
print(f'\n' + '=' * 80)
print('SUMMARY')
print('=' * 80)
print(f'Total videos processed: {len(all_results)}')
print(f'Successful: {success_count}')
print(f'Failed: {failure_count}')
print(f'Success rate: {success_count/len(all_results)*100:.1f}%')
print(f'PPG Sync available: {ppg_sync_count}/{success_count}')
print(f'\nOutput saved to: {config.OUTPUT_ROOT}')
print(f'Faces saved to: {config.FACES_PATH}')
print(f'Landmarks saved to: {config.LANDMARKS_PATH}')
print(f'PPG Sync saved to: {config.PPG_SYNC_PATH}')
print(f'\nTraining CSV: {csv_file}')
print(f'Simplified training CSV: {simple_csv_file}')

print('\n' + '=' * 80)
print('READY FOR TRAINING with rppglib.train.train_fold!')
print('=' * 80)

---

## 15. Quick Test: Load and Verify One Sample

In [ ]:
# Load and verify one sample from the preprocessed data
print('\nQuick verification test...')

# Get first successful result with PPG sync
first_success = None
for result in all_results:
    if result['success'] and result['ppg_sync'] is not None:
        first_success = result
        break

if first_success:
    video_id = os.path.splitext(os.path.basename(first_success['video_path']))[0]
    
    # Load saved data
    landmarks_file = os.path.join(config.LANDMARKS_PATH, f'{video_id}_landmarks.npy')
    full_face_file = os.path.join(config.OUTPUT_ROOT, 'full_face', f'{video_id}.npy')
    forehead_file = os.path.join(config.OUTPUT_ROOT, 'forehead', f'{video_id}.npy')
    ppg_sync_file = os.path.join(config.PPG_SYNC_PATH, f'{video_id}.npy')
    npz_file = os.path.join(config.OUTPUT_ROOT, 'train_data', f'{video_id}.npz')
    
    print(f'Video ID: {video_id}')
    
    if os.path.exists(landmarks_file):
        loaded_landmarks = np.load(landmarks_file)
        print(f'✓ Landmarks loaded: {loaded_landmarks.shape}')
    else:
        print(f'✗ Landmarks not found: {landmarks_file}')
    
    if os.path.exists(full_face_file):
        loaded_face = np.load(full_face_file)
        print(f'✓ Full face video loaded: {loaded_face.shape}')
    else:
        print(f'✗ Full face not found: {full_face_file}')
    
    if os.path.exists(forehead_file):
        loaded_forehead = np.load(forehead_file)
        print(f'✓ Forehead ROI loaded: {loaded_forehead.shape}')
    else:
        print(f'✗ Forehead ROI not found: {forehead_file}')
    
    if os.path.exists(ppg_sync_file):
        loaded_ppg_sync = np.load(ppg_sync_file)
        print(f'✓ PPG Sync signal loaded: {loaded_ppg_sync.shape}')
    else:
        print(f'✗ PPG Sync signal not found: {ppg_sync_file}')
    
    if os.path.exists(npz_file):
        loaded_npz = np.load(npz_file)
        print(f'✓ NPZ file loaded: {npz_file}')
        print(f'  Keys: {list(loaded_npz.keys())}')
        print(f'  Video shape: {loaded_npz["video"].shape}')
        print(f'  PPG shape: {loaded_npz["ppg"].shape}')
        print(f'  Landmarks shape: {loaded_npz["landmarks"].shape}')
    else:
        print(f'✗ NPZ file not found: {npz_file}')
    
    # Display a sample frame
    if os.path.exists(full_face_file):
        plt.figure(figsize=(10, 5))
        plt.subplot(1, 2, 1)
        plt.imshow(loaded_face[0])
        plt.title(f'Full Face - Frame 0')
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        if os.path.exists(forehead_file):
            plt.imshow(loaded_forehead[0])
            plt.title(f'Forehead ROI - Frame 0')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    
    # Plot PPG sync signal
    if os.path.exists(ppg_sync_file):
        plt.figure(figsize=(12, 4))
        plt.plot(loaded_ppg_sync[:200])
        plt.title(f'PPG Sync Signal - First 200 frames (ALIGNED with video)')
        plt.xlabel('Frame')
        plt.ylabel('PPG Sync Value')
        plt.grid(True)
        plt.show()
    
    print('\n✓ Verification test passed!')
    print('\n✓ Data is ready for rppglib.train.train_fold!')
else:
    print('No successful results with PPG sync to verify')

---

## Notebook Complete!

The preprocessing pipeline has been completed. All data has been saved to:
- `/home/cristic/preprocessed_data/` - Root output directory
- `/home/cristic/preprocessed_data/faces/` - Processed face videos
- `/home/cristic/preprocessed_data/landmarks/` - Face landmarks
- `/home/cristic/preprocessed_data/ppg_sync/` - PPG sync signals (ALREADY ALIGNED with video)
- `/home/cristic/preprocessed_data/{roi_name}/` - ROI videos for each ROI
- `/home/cristic/preprocessed_data/ppg/` - Aligned PPG signals (from .PW files)
- `/home/cristic/preprocessed_data/train_data/` - .npz files for training

### Files Created:
- `metadata.csv` - Metadata for all processed videos
- `ppg_metadata.csv` - Metadata for PPG signals
- `ppg_sync_metadata.csv` - Metadata for PPG sync signals
- `train_videos.txt`, `val_videos.txt`, `test_videos.txt` - Dataset splits
- `{roi_name}_all.npy` - Combined ROI arrays
- `mcd_rppg.csv` - Training CSV for rppglib.train.train_fold (FULL FORMAT)
- `train_folds.csv` - Simplified training CSV with file and fold columns

### Training with rppglib:

```python
import rppglib.train
from rppglib.params import Params

# Load config
config = Params()
config.train_dataset = '/home/cristic/preprocessed_data/mcd_rppg'  # without .csv
config.test_datasets = ['/home/cristic/preprocessed_data/mcd_rppg']
config.model = 'SCNN_8rois'
config.valid_fold = 0
config.test_fold = 0

# Train
config = rppglib.train.train_fold(config)
```

### Next Steps:
1. Verify the output files exist and have correct shapes
2. Use the preprocessed data for training rPPG models
3. Optionally adjust ROI definitions or preprocessing parameters

### Key Notes:
- **PPG Sync files are ALREADY ALIGNED with video frames** - no alignment needed!
- Use `mcd_rppg.csv` or `train_folds.csv` with `rppglib.train.train_fold`
- .npz files contain: video, ppg (from ppg_sync), landmarks
- PPG sync is at video frame rate (30 FPS), not PPG sensor rate (100 Hz)
- Notebook includes fallback implementations for API compatibility

### IMPORTANT: Execution Order
**MUST execute cells in order!** The notebook has dependencies between cells:
1. Run all import and configuration cells first (Cells 1-4)
2. Run file parsing functions (Cell 5)
3. **CRITICAL: Run ROI extraction functions (Cell 6) BEFORE video processing**
4. Run PPG alignment functions (Cell 7)
5. Run video processing functions (Cell 8)
6. Then proceed with TEST PHASE and full processing

If you get 'NameError: name X is not defined', you skipped a required cell!